# Projeto Prático 2 (Parte 2) — Classificação Industrial com `Softmax` e `Cross-Entropy`

Na **[Parte 1 (`04_projeto2_pmc_conservantes_numpy.ipynb`)](04_projeto2_pmc_conservantes_numpy.ipynb)** treinamos nossa rede `4-15-3` manualmente com Sigmóide e Erro Quadrático Médio (`MSE`).

Nas ferramentas modernas da indústria (`Scikit-Learn` e `Keras`), problemas de classificação multiclasse utilizam uma dupla matematicamente ainda mais poderosa na camada de saída:
1. **Ativação `Softmax`:** Converte os potenciais $[u_1, u_2, u_3]$ em **probabilidades reais** que somam exatamente `1,0` (`100%`), como `"92% Conservante A, 7% B, 1% C"`.
2. **Função de Custo `Cross-Entropy` (Entropia Cruzada):** Penaliza logaritmicamente previsões confiantes e erradas, evitando a lentidão da derivada saturada da sigmóide!

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

train_df = pd.read_csv("datasets/projeto-pratico-2.csv")
test_df = pd.read_csv("datasets/conjunto-testes-projeto-pratico-2.csv")

X_train = train_df[["x1", "x2", "x3", "x4"]].to_numpy()
D_train = train_df[["d1", "d2", "d3"]].to_numpy()
y_train_idx = np.argmax(D_train, axis=1)  # Converte One-Hot [1,0,0],[0,1,0],[0,0,1] -> 0, 1, 2

X_test = test_df[["x1", "x2", "x3", "x4"]].to_numpy()
D_test = test_df[["d1", "d2", "d3"]].to_numpy()
y_test_idx = np.argmax(D_test, axis=1)

print("Distribuição de conservantes no treino (0=A, 1=B, 2=C):", np.bincount(y_train_idx))

---

## 1. Resolução com `MLPClassifier` (`Scikit-Learn`) e Matriz de Confusão

Quando passamos rótulos `0, 1, 2` (mutuamente exclusivos) ao `MLPClassifier`, ele ativa automaticamente a camada final **`Softmax`**! Além da acurácia, exibimos a **Matriz de Confusão** (tabela cruzada onde a diagonal principal mostra os acertos de cada conservante `A`, `B` e `C`).

In [ ]:
# solver='lbfgs' converge sem avisos em poucas dezenas de passos para N=148 amostras!
sk_clf = MLPClassifier(
    hidden_layer_sizes=(15,),
    activation="logistic",
    solver="lbfgs",
    max_iter=2000,
    random_state=42,
)
sk_clf.fit(X_train, y_train_idx)

y_pred_sk = sk_clf.predict(X_test)
acc_sk = accuracy_score(y_test_idx, y_pred_sk) * 100.0

print(f"Acurácia no Teste (Scikit-Learn + Softmax): {acc_sk:.2f}% (em {sk_clf.n_iter_} iterações)")
print("\nMatriz de Confusão (Linhas = Real [A,B,C] | Colunas = Previsto [A,B,C]):")
print(confusion_matrix(y_test_idx, y_pred_sk))

---

## 2. Resolução com `Keras` (`TensorFlow`): `ReLU` + `Softmax` + `Adam`

No `Keras`, montamos explicitamente as camadas usando `keras.Input(shape=(4,))` (sem avisos de depreciação!), ativação `relu` nas camadas internas e `softmax` na camada final com perda `categorical_crossentropy` (que recebe diretamente nossa matriz *One-Hot* `D_train`).

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)

modelo_keras = keras.Sequential([
    keras.Input(shape=(4,)),                       # 4 sensores físico-químicos de entrada
    keras.layers.Dense(24, activation="relu"),     # 1ª camada oculta com ReLU
    keras.layers.Dense(12, activation="relu"),     # 2ª camada oculta com ReLU
    keras.layers.Dense(3, activation="softmax"),   # 3 probabilidades que somam 100%
])

modelo_keras.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

modelo_keras.fit(X_train, D_train, epochs=250, batch_size=16, verbose=0)

# Probabilidades previstas para cada conservante (A, B, C) nas 18 amostras de teste
probabilidades = modelo_keras(X_test, training=False).numpy()
y_pred_keras = np.argmax(probabilidades, axis=1)
acc_keras = accuracy_score(y_test_idx, y_pred_keras) * 100.0

print(f"Acurácia no Teste (Keras Deep Learning): {acc_keras:.2f}%")
print("\nProbabilidades nas 5 primeiras amostras de teste [P(A), P(B), P(C)]:")
print(np.round(probabilidades[:5], 4))

**Leitura das Probabilidades:** Em controle industrial automatizado, ter `[0.001, 0.004, 0.995]` é muito mais rico do que apenas `C`: se a maior probabilidade cair abaixo de `70%`, o sistema supervisório (SCADA) pode alertar um engenheiro químico para inspecionar o lote manualmente!

---

## 3. Exercícios Práticos de Fixação

### Exercício 1: Consultando Probabilidades no `Scikit-Learn` (`predict_proba`)
O `MLPClassifier` também fornece probabilidades via `sk_clf.predict_proba(X_test)`! Verifique que a soma de cada linha de `sk_clf.predict_proba(X_test)` é exatamente `1.0`.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
probs_sk = sk_clf.predict_proba(X_test)
print("Primeiras 3 linhas de probabilidades:", np.round(probs_sk[:3], 4))
print("Soma das probabilidades por amostra:", np.round(probs_sk.sum(axis=1), 6))
```

</details>

### Exercício 2: Identificando a Amostra de Menor Confiança
Descubra qual das 18 amostras de teste teve a **menor probabilidade máxima** segundo `probs_sk` (isto é, o ensaio em que a rede ficou mais em dúvida entre dois conservantes).

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
probs_sk = sk_clf.predict_proba(X_test)
confianca_maxima = np.max(probs_sk, axis=1)
idx_menor = int(np.argmin(confianca_maxima))

print(f"Amostra mais incerta: Amostra #{int(test_df.loc[idx_menor, 'Amostra'])}")
print(f"Confiança máxima: {confianca_maxima[idx_menor] * 100:.2f}% | Distribuição: {np.round(probs_sk[idx_menor], 4)}")
```

</details>